# Comparing three views of downside risk

## Comparing value at risk with expected shortfall

For loss \(L\) and probability level \(p\in(0,1)\), value at risk is the lower quantile boundary

$$
\operatorname{VaR}_p(L)=\inf\{x:\Pr(L\leq x)\geq p\}.
$$

The value \(p\) is the covered probability, \(1-p\) is the tail probability, and \(\operatorname{VaR}_p(L)\) is a loss threshold. A precise expected shortfall definition averages the quantile function over the upper tail:

$$
\operatorname{ES}_p(L)=\frac{1}{1-p}\int_p^1\operatorname{VaR}_q(L)\,dq.
$$

Here, \(q\) is the probability level being averaged. This integral remains meaningful when probability mass lies at the value at risk boundary. For continuous losses, it agrees with the mean loss beyond that boundary.

For a generalized Pareto tail with fitted loss quantile \(q_p>u\), expected shortfall is

$$
\operatorname{ES}_p(L)=q_p+\frac{\beta+\xi(q_p-u)}{1-\xi},
$$

provided \(\xi<1\). The threshold is \(u\), the scale is \(\beta\), and the shape is \(\xi\). When \(\xi\geq 1\), the fitted tail has no finite mean, so this expected shortfall is infinite even if the fitted quantile is finite.

In [1]:
from pathlib import Path
import sys
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import HTML, display as _display

def display(value):
    module = value.__class__.__module__
    if module.startswith('pandas') and hasattr(value, '_repr_html_'):
        html = value._repr_html_()
        return _display(HTML('<div style="display:flex;justify-content:center;">' + html + '</div>'))
    return _display(value)
from scipy.stats import genpareto, norm
BOOK_ROOT = Path.cwd().parents[1]
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))
from book_support import ensure_dataset
import RiskLabAI
from RiskLabAI.backtest.backtest_overfitting_simulation import expected_shortfall
from RiskLabAI.backtest.robust_statistics import conditional_expected_drawdown
from RiskLabAI.backtest.tail_risk import fit_pot_tail_risk, generalized_pareto_tail_risk, quantile_exception_tests
assert RiskLabAI.__version__ == '3.2.0'
DATA_PATH = ensure_dataset(BOOK_ROOT, 'fred_nasdaq')
FIGURES = BOOK_ROOT / 'outputs' / 'figures' / '21_tail_risk'
TABLES = BOOK_ROOT / 'outputs' / 'tables' / '21_tail_risk'
FIGURE_DATA = BOOK_ROOT / 'outputs' / 'figure_data' / '21_tail_risk'
for directory in [FIGURES, TABLES, FIGURE_DATA]:
    directory.mkdir(parents=True, exist_ok=True)
PALETTE = {'background': '#111827', 'axes': '#111827', 'text': '#E5E7EB', 'cyan': '#38BDF8', 'amber': '#FBBF24', 'green': '#34D399', 'rose': '#FB7185', 'violet': '#C084FC', 'grid': '#374151'}
plt.rcParams.update({'font.family': 'DejaVu Serif', 'font.size': 11.5, 'figure.facecolor': PALETTE['background'], 'axes.facecolor': PALETTE['axes'], 'axes.edgecolor': PALETTE['text'], 'axes.labelcolor': PALETTE['text'], 'axes.titlecolor': PALETTE['text'], 'xtick.color': PALETTE['text'], 'ytick.color': PALETTE['text'], 'text.color': PALETTE['text'], 'grid.color': PALETTE['grid'], 'grid.linestyle': ':', 'legend.facecolor': PALETTE['axes'], 'legend.edgecolor': PALETTE['grid'], 'savefig.facecolor': PALETTE['background'], 'savefig.transparent': False, 'savefig.dpi': 300})

def rendered_value(value):
    if pd.isna(value):
        return ''
    if isinstance(value, (bool, np.bool_)):
        return 'Pass' if bool(value) else 'Fail'
    if isinstance(value, (pd.Timestamp, np.datetime64)):
        return pd.Timestamp(value).date().isoformat()
    if isinstance(value, (int, np.integer)):
        return f'{int(value):,}'
    if isinstance(value, (float, np.floating)):
        number = float(value)
        if np.isposinf(number):
            return 'Infinite'
        if np.isneginf(number):
            return 'Negative infinite'
        if number == 0:
            return '0'
        if number.is_integer():
            return f'{int(number):,}'
        magnitude = abs(number)
        if magnitude < 0.0001 or magnitude >= 1000000:
            return f'{number:.6g}'
        return f'{number:.6f}'.rstrip('0').rstrip('.')
    return str(value)

def rendered_frame(frame):
    return frame.apply(lambda column: column.map(rendered_value))

def save_table(frame, stem, caption, label, fit=False):
    shown = rendered_frame(frame)
    shown.to_csv(TABLES / f'{stem}.csv', index=False)
    latex = shown.to_latex(index=False, escape=True, caption=caption, label=label, position='htbp')
    if fit:
        latex = latex.replace('\\begin{tabular}', '\\begin{adjustbox}{max width=\\textwidth}\\begin{tabular}', 1).replace('\\end{tabular}', '\\end{tabular}\\end{adjustbox}', 1)
    (TABLES / f'{stem}.tex').write_text(latex, encoding='utf-8')
    display(shown)
    return shown

def finish_axis(axis, grid=True):
    axis.spines['top'].set_visible(False)
    axis.spines['right'].set_visible(False)
    if grid:
        axis.grid(True, alpha=0.65)
    axis.set_axisbelow(True)

def save_figure(figure, stem):
    figure.savefig(FIGURES / f'{stem}.png', dpi=300, bbox_inches='tight', facecolor=PALETTE['background'], transparent=False)
    figure.savefig(FIGURES / f'{stem}.pdf', bbox_inches='tight', facecolor=PALETTE['background'], transparent=False)

def load_public_data():
    assert DATA_PATH.is_file()
    raw = pd.read_csv(DATA_PATH)
    raw['Date'] = pd.to_datetime(raw['observation_date'])
    raw['NASDAQ Composite'] = pd.to_numeric(raw['NASDAQCOM'], errors='coerce')
    data = raw.dropna(subset=['NASDAQ Composite']).sort_values('Date').reset_index(drop=True)
    data['Daily log return (%)'] = 100 * np.log(data['NASDAQ Composite'] / data['NASDAQ Composite'].shift(1))
    data['Loss (%)'] = -data['Daily log return (%)']
    data['Simple return'] = np.expm1(data['Daily log return (%)'] / 100)
    return data

def split_samples(data):
    returns = data.dropna(subset=['Loss (%)']).copy()
    design = returns[returns['Date'] <= pd.Timestamp('2014-12-31')].copy()
    held = returns[returns['Date'] >= pd.Timestamp('2015-01-01')].copy()
    assert len(design) == 5287
    assert len(held) == 2767
    return (returns, design, held)

def empirical_upper_tail_mean(losses, probability):
    values = np.sort(np.asarray(losses, dtype=float))
    count = (1 - probability) * len(values)
    full = int(np.floor(count))
    total = values[len(values) - full:].sum() if full > 0 else 0.0
    fraction = count - full
    if fraction > 0:
        total += fraction * values[len(values) - full - 1]
    return float(total / count)

def rolling_max_drawdowns(simple_returns, horizon):
    wealth = np.cumprod(1 + np.asarray(simple_returns, dtype=float))
    if len(wealth) < horizon or horizon < 2:
        drawdown = 1 - wealth / np.maximum.accumulate(wealth)
        return np.array([drawdown.max() if len(drawdown) else 0.0])
    windows = np.lib.stride_tricks.sliding_window_view(wealth, horizon)
    high_water = np.maximum.accumulate(windows, axis=1)
    return (1 - windows / high_water).max(axis=1)

def maximum_drawdown(simple_returns):
    wealth = np.cumprod(1 + np.asarray(simple_returns, dtype=float))
    drawdown = 1 - wealth / np.maximum.accumulate(wealth)
    return float(drawdown.max())

def moving_block_sample(values, length, block, rng):
    array = np.asarray(values, dtype=float)
    starts = rng.integers(0, len(array) - block + 1, size=int(np.ceil(length / block)))
    return np.concatenate([array[start:start + block] for start in starts])[:length]

def annual_pot_forecast(all_returns, held, probability):
    forecasts = pd.Series(index=held.index, dtype=float)
    records = []
    for year in sorted(held['Date'].dt.year.unique()):
        start = pd.Timestamp(f'{year}-01-01')
        history = all_returns[all_returns['Date'] < start]
        threshold = float(history['Loss (%)'].quantile(0.95))
        fit = fit_pot_tail_risk(history['Loss (%)'].to_numpy(), threshold=threshold, probability=probability)
        mask = held['Date'].dt.year == year
        forecasts.loc[held.index[mask]] = fit['value_at_risk']
        records.append({'Forecast year': int(year), 'History through': history['Date'].max(), 'History observations': int(len(history)), 'Threshold (%)': threshold, 'Excesses': int(fit['exceedance_count']), 'Shape': fit['shape'], 'Scale (%)': fit['scale'], 'Forecast VaR (%)': fit['value_at_risk'], 'Forecast ES (%)': fit['expected_shortfall']})
    assert forecasts.notna().all()
    return (forecasts, pd.DataFrame(records))
data = load_public_data()
returns, design, held = split_samples(data)
losses = design['Loss (%)'].to_numpy()
return_values = design['Daily log return (%)'].to_numpy()
threshold = float(np.quantile(losses, 0.95))
pot = fit_pot_tail_risk(losses, threshold=threshold, probability=0.99)
mu = float(losses.mean())
sigma = float(losses.std(ddof=1))
definitions = pd.DataFrame([{'Measure': 'Value at risk', 'Question': 'What loss threshold has the stated upper-tail probability?', 'Needs': 'Probability level'}, {'Measure': 'Expected shortfall', 'Question': 'What is the average loss in the stated upper tail?', 'Needs': 'Probability level'}, {'Measure': 'Maximum drawdown', 'Question': 'What is the largest peak-to-trough fall on a path?', 'Needs': 'Path'}, {'Measure': 'Conditional expected drawdown', 'Question': 'What is the mean of the worst horizon drawdowns?', 'Needs': 'Horizon and tail level'}])
save_table(definitions, 'risk_measure_definitions', 'Four risk measures answer different questions.', 'tab:chapter22-risk-definitions', fit=True)
risk_rows = []
for probability in [0.95, 0.975, 0.99, 0.995]:
    z = norm.ppf(probability)
    gaussian_var = mu + sigma * z
    gaussian_es = mu + sigma * norm.pdf(z) / (1 - probability)
    empirical_var = float(np.quantile(losses, probability))
    empirical_es = -float(expected_shortfall(return_values, 0.0, confidence_level=1 - probability))
    risk_rows.extend([{'Probability level': probability, 'Tail probability': 1 - probability, 'Method': 'Gaussian', 'VaR (%)': gaussian_var, 'ES (%)': gaussian_es}, {'Probability level': probability, 'Tail probability': 1 - probability, 'Method': 'Empirical', 'VaR (%)': empirical_var, 'ES (%)': empirical_es}])
    if probability > 0.95:
        tail = generalized_pareto_tail_risk(threshold, pot['exceedance_probability'], pot['shape'], pot['scale'], probability)
        risk_rows.append({'Probability level': probability, 'Tail probability': 1 - probability, 'Method': 'Generalized Pareto', 'VaR (%)': tail['value_at_risk'], 'ES (%)': tail['expected_shortfall']})
risk_table = pd.DataFrame(risk_rows)
save_table(risk_table, 'var_es_comparison', 'Design-sample value at risk and expected shortfall estimates.', 'tab:chapter22-var-es', fit=True)
curve_rows = []
for probability in np.linspace(0.955, 0.999, 70):
    z = norm.ppf(probability)
    curve_rows.append({'Probability level': probability, 'Method': 'Gaussian', 'VaR (%)': mu + sigma * z, 'ES (%)': mu + sigma * norm.pdf(z) / (1 - probability)})
    curve_rows.append({'Probability level': probability, 'Method': 'Empirical', 'VaR (%)': float(np.quantile(losses, probability)), 'ES (%)': empirical_upper_tail_mean(losses, probability)})
    tail = generalized_pareto_tail_risk(threshold, pot['exceedance_probability'], pot['shape'], pot['scale'], probability)
    curve_rows.append({'Probability level': probability, 'Method': 'Generalized Pareto', 'VaR (%)': tail['value_at_risk'], 'ES (%)': tail['expected_shortfall']})
curve = pd.DataFrame(curve_rows)
curve.to_csv(FIGURE_DATA / 'var_expected_shortfall_curves.csv', index=False)
figure, axes = plt.subplots(1, 2, figsize=(11.2, 5.0), constrained_layout=True)
colors = {'Gaussian': PALETTE['amber'], 'Empirical': PALETTE['green'], 'Generalized Pareto': PALETTE['cyan']}
for method, part in curve.groupby('Method'):
    axes[0].plot(part['Probability level'], part['VaR (%)'], color=colors[method], linewidth=1.8, label=method)
    axes[1].plot(part['Probability level'], part['ES (%)'], color=colors[method], linewidth=1.8, label=method)
axes[0].set_title('Value at Risk')
axes[1].set_title('Expected Shortfall')
for axis in axes:
    axis.set_xlabel('Probability level')
    axis.set_ylabel('Loss (%)')
    axis.legend()
    finish_axis(axis)
save_figure(figure, 'var_expected_shortfall_curves')
plt.close(figure)
probability = 0.99
empirical_var = float(np.quantile(losses, probability))
empirical_es = empirical_upper_tail_mean(losses, probability)
ordered = np.sort(losses)[::-1][:350]
contributions = pd.DataFrame({'Worst-loss rank': np.arange(1, len(ordered) + 1), 'Loss (%)': ordered})
contributions['At or above 99% VaR'] = contributions['Loss (%)'] >= empirical_var
contributions['99% VaR (%)'] = empirical_var
contributions['99% ES (%)'] = empirical_es
contributions.to_csv(FIGURE_DATA / 'tail_loss_contributions.csv', index=False)
figure, axis = plt.subplots(figsize=(10.8, 5.4), constrained_layout=True)
colors_rank = np.where(contributions['At or above 99% VaR'], PALETTE['rose'], PALETTE['cyan'])
axis.scatter(contributions['Worst-loss rank'], contributions['Loss (%)'], c=colors_rank, s=16, alpha=0.8)
axis.axhline(empirical_var, color=PALETTE['amber'], linestyle='--', linewidth=1.4, label='99% value at risk')
axis.axhline(empirical_es, color=PALETTE['green'], linestyle=':', linewidth=1.8, label='99% expected shortfall')
axis.set_xlabel('Rank among the largest design losses')
axis.set_ylabel('Daily loss (%)')
axis.set_title('Value at Risk Is a Boundary, Expected Shortfall Is a Tail Mean')
axis.legend()
finish_axis(axis)
save_figure(figure, 'tail_loss_contributions')
plt.close(figure)
boundary_rows = []
for shape in [-0.25, 0.0, 0.25, 0.75, 1.0, 1.25]:
    result = generalized_pareto_tail_risk(threshold, pot['exceedance_probability'], shape, pot['scale'], 0.99)
    boundary_rows.append({'Shape': shape, '99% VaR (%)': result['value_at_risk'], '99% ES (%)': result['expected_shortfall'], 'Finite expected shortfall': np.isfinite(result['expected_shortfall'])})
boundary = pd.DataFrame(boundary_rows)
save_table(boundary, 'finite_mean_boundary', 'The generalized Pareto finite-mean boundary.', 'tab:chapter22-finite-mean')
assert np.isclose(empirical_es, -expected_shortfall(return_values, 0.0, confidence_level=0.01), atol=0.02)
assert all(risk_table['ES (%)'] >= risk_table['VaR (%)'])
assert not boundary.loc[boundary['Shape'] >= 1, 'Finite expected shortfall'].any()


,Measure,Question,Needs
0,Value at risk,What loss threshold has the stated upper-tail ...,Probability level
1,Expected shortfall,What is the average loss in the stated upper t...,Probability level
2,Maximum drawdown,What is the largest peak-to-trough fall on a p...,Path
3,Conditional expected drawdown,What is the mean of the worst horizon drawdowns?,Horizon and tail level


,Probability level,Tail probability,Method,VaR (%),ES (%)
0,0.95,0.05,Gaussian,2.596823,3.265245
1,0.95,0.05,Empirical,2.614336,3.810421
2,0.975,0.025,Gaussian,3.100885,3.705289
3,0.975,0.025,Empirical,3.303977,4.684206
4,0.975,0.025,Generalized Pareto,3.370224,4.688345
5,0.99,0.01,Gaussian,3.686965,4.229027
6,0.99,0.01,Empirical,4.4378,6.00047
7,0.99,0.01,Generalized Pareto,4.481563,5.974601
8,0.995,0.005,Gaussian,4.086044,4.591719
9,0.995,0.005,Empirical,5.589122,7.003071


,Shape,99% VaR (%),99% ES (%),Finite expected shortfall
0,-0.25,3.988883,4.542837,Pass
1,0,4.284386,5.320466,Pass
2,0.25,4.671025,6.738027,Pass
3,0.75,5.860533,19.743442,Pass
4,1,6.771392,Infinite,Fail
5,1.25,8.001727,Infinite,Fail


## Drawdown as a Path Quantity

For wealth \(W_t\), drawdown at date \(t\) is

$$
D_t=1-\frac{W_t}{\max_{s\leq t}W_s},
$$

where the denominator is the highest wealth reached through date \(t\). Maximum drawdown over a path is \(\max_t D_t\). For a fixed horizon \(h\), let \(M_h\) denote maximum drawdown within a horizon window. Conditional expected drawdown at level \(a\) is

$$
\operatorname{CED}_{a,h}=\frac{1}{1-a}\int_a^1\operatorname{VaR}_q(M_h)\,dq.
$$

The horizon \(h\) is measured in trading days, \(a\) is the covered probability, and \(M_h\) is a distribution of path losses rather than one-day losses. The reported value must therefore state both the horizon and the tail level.

In [2]:
data = load_public_data()
returns, design, held = split_samples(data)
simple = returns["Simple return"].to_numpy()
wealth = np.cumprod(1 + simple)
drawdown = 1 - wealth / np.maximum.accumulate(wealth)
drawdown_history = pd.DataFrame({"Date": returns["Date"].to_numpy(), "Wealth from one unit": wealth, "Drawdown": drawdown})
drawdown_history.to_csv(FIGURE_DATA / "wealth_drawdown_history.csv", index=False)
figure, axes = plt.subplots(2, 1, figsize=(11.2, 7.0), sharex=True, constrained_layout=True)
axes[0].plot(drawdown_history["Date"], drawdown_history["Wealth from one unit"], color=PALETTE["cyan"], linewidth=1.15)
axes[0].set_yscale("log")
axes[0].set_ylabel("Wealth from one unit")
axes[0].set_title("Public Wealth Path and Drawdown")
axes[1].fill_between(drawdown_history["Date"], 0, drawdown_history["Drawdown"], color=PALETTE["rose"], alpha=0.75)
axes[1].set_ylabel("Drawdown")
axes[1].set_xlabel("Date")
for axis in axes:
    finish_axis(axis)
save_figure(figure, "wealth_drawdown_history")
plt.close(figure)

drawdown_definitions = pd.DataFrame([
    {"Quantity": "Wealth", "Construction": "Cumulative product of one plus each simple return", "Unit": "Relative wealth"},
    {"Quantity": "Drawdown", "Construction": "One minus wealth divided by its running maximum", "Unit": "Fraction"},
    {"Quantity": "Maximum drawdown", "Construction": "Largest drawdown on a path or window", "Unit": "Fraction"},
    {"Quantity": "Conditional expected drawdown", "Construction": "Upper-tail mean of horizon maximum drawdowns", "Unit": "Fraction"},
])
save_table(drawdown_definitions, "drawdown_definitions", "Definitions for path-based loss measures.", "tab:chapter22-drawdown-definitions", fit=True)

sensitivity_rows = []
for horizon in [20, 60, 126, 252]:
    maxima = rolling_max_drawdowns(simple, horizon)
    for level in [0.90, 0.95, 0.975]:
        sensitivity_rows.append({"Horizon in trading days": horizon, "Tail level": level, "Rolling windows": len(maxima), "Conditional expected drawdown": conditional_expected_drawdown(simple, horizon=horizon, alpha=level), "Largest window drawdown": maxima.max()})
drawdown_sensitivity = pd.DataFrame(sensitivity_rows)
save_table(drawdown_sensitivity, "drawdown_sensitivity", "Conditional expected drawdown across horizons and tail levels.", "tab:chapter22-drawdown-sensitivity", fit=True)

heat = drawdown_sensitivity[["Horizon in trading days", "Tail level", "Conditional expected drawdown"]].copy()
heat.to_csv(FIGURE_DATA / "drawdown_sensitivity_heatmap.csv", index=False)
matrix = heat.pivot(index="Horizon in trading days", columns="Tail level", values="Conditional expected drawdown")
figure, axis = plt.subplots(figsize=(7.8, 5.6), constrained_layout=True)
image = axis.imshow(matrix.to_numpy(), aspect="auto", cmap="magma")
axis.set_xticks(range(len(matrix.columns)), [f"{value:g}" for value in matrix.columns])
axis.set_yticks(range(len(matrix.index)), [str(value) for value in matrix.index])
axis.set_xlabel("Tail level")
axis.set_ylabel("Horizon in trading days")
axis.set_title("Conditional Expected Drawdown Depends on Two Settings")
for i in range(len(matrix.index)):
    for j in range(len(matrix.columns)):
        axis.text(j, i, f"{matrix.iloc[i, j]:.3f}", ha="center", va="center", color="white")
colorbar = figure.colorbar(image, ax=axis)
colorbar.set_label("Conditional expected drawdown")
save_figure(figure, "drawdown_sensitivity_heatmap")
plt.close(figure)

assert np.all(drawdown >= 0)
assert np.all(drawdown < 1)
assert all(drawdown_sensitivity["Conditional expected drawdown"] <= drawdown_sensitivity["Largest window drawdown"] + 1e-12)

,Quantity,Construction,Unit
0,Wealth,Cumulative product of one plus each simple return,Relative wealth
1,Drawdown,One minus wealth divided by its running maximum,Fraction
2,Maximum drawdown,Largest drawdown on a path or window,Fraction
3,Conditional expected drawdown,Upper-tail mean of horizon maximum drawdowns,Fraction


,Horizon in trading days,Tail level,Rolling windows,Conditional expected drawdown,Largest window drawdown
0,20,0.9,"8,035",0.155845,0.330794
1,20,0.95,"8,035",0.189706,0.330794
2,20,0.975,"8,035",0.223604,0.330794
3,60,0.9,"7,995",0.276772,0.454263
4,60,0.95,"7,995",0.329062,0.454263
5,60,0.975,"7,995",0.365959,0.454263
6,126,0.9,"7,929",0.383088,0.534841
7,126,0.95,"7,929",0.434595,0.534841
8,126,0.975,"7,929",0.475784,0.534841
9,252,0.9,"7,803",0.500277,0.63484


## Public-Data Block Resampling

A moving-block resample joins blocks of \(b\) consecutive design returns until a path of length \(m\) is filled:

$$
R^{*(j)}=\bigl(R_{S_1:S_1+b-1},R_{S_2:S_2+b-1},\ldots\bigr)_{1:m}.
$$

Here, \(S_1,S_2,\ldots\) are independently drawn block starting positions, \(b=20\) is the block length, \(m=500\) is the resampled path length, and \(j\) identifies one of 300 resamples. Blocks preserve short runs from the public design sample. They do not create new market observations or a population truth.

For a positive statistic \(X\), its coefficient of variation is

$$
\operatorname{CV}(X)=\frac{\operatorname{sd}(X)}{\mathbb{E}[X]}.
$$

The standard deviation measures dispersion across resamples and the mean rescales it. This experiment compares the sampling variation of maximum drawdown and conditional expected drawdown only under the declared public-data resampling design.

In [3]:
data = load_public_data()
returns, design, held = split_samples(data)
rng = np.random.default_rng(2202)
settings = {"Resamples": 300, "Path length": 500, "Block length": 20, "Drawdown horizon": 60, "CED tail level": 0.95, "Random seed": 2202}
rows = []
design_returns = design["Simple return"].to_numpy()
for replicate in range(1, settings["Resamples"] + 1):
    sample = moving_block_sample(design_returns, settings["Path length"], settings["Block length"], rng)
    rows.append({"Resample": replicate, "Maximum drawdown": maximum_drawdown(sample), "Conditional expected drawdown": conditional_expected_drawdown(sample, horizon=settings["Drawdown horizon"], alpha=settings["CED tail level"])})
resampled = pd.DataFrame(rows)
resampled.to_csv(FIGURE_DATA / "resampled_drawdown_measures.csv", index=False)

settings_table = pd.DataFrame([{"Setting": key, "Value": value} for key, value in settings.items()])
save_table(settings_table, "resampling_settings", "Fixed public-data resampling settings.", "tab:chapter22-resampling-settings")

summary_rows = []
for column in ["Maximum drawdown", "Conditional expected drawdown"]:
    values = resampled[column]
    summary_rows.append({"Measure": column, "Mean": values.mean(), "Standard deviation": values.std(ddof=1), "Coefficient of variation": values.std(ddof=1) / values.mean(), "Minimum": values.min(), "Maximum": values.max()})
resampling_summary = pd.DataFrame(summary_rows)
save_table(resampling_summary, "resampling_summary", "Sampling variation under the fixed block-resampling design.", "tab:chapter22-resampling-summary", fit=True)

quantile_rows = []
for column in ["Maximum drawdown", "Conditional expected drawdown"]:
    for probability in [0.05, 0.25, 0.50, 0.75, 0.95]:
        quantile_rows.append({"Measure": column, "Probability": probability, "Quantile": resampled[column].quantile(probability)})
resampling_quantiles = pd.DataFrame(quantile_rows)
save_table(resampling_quantiles, "resampling_quantiles", "Quantiles across public-data block resamples.", "tab:chapter22-resampling-quantiles")

figure, axis = plt.subplots(figsize=(10.6, 5.4), constrained_layout=True)
bins = np.linspace(min(resampled.min(numeric_only=True)), max(resampled.max(numeric_only=True)), 32)
axis.hist(resampled["Maximum drawdown"], bins=bins, alpha=0.62, color=PALETTE["amber"], label="Maximum drawdown")
axis.hist(resampled["Conditional expected drawdown"], bins=bins, alpha=0.62, color=PALETTE["cyan"], label="Conditional expected drawdown")
axis.set_xlabel("Drawdown measure")
axis.set_ylabel("Resamples")
axis.set_title("Public-Data Block Resampling Distributions")
axis.legend()
finish_axis(axis)
save_figure(figure, "resampled_drawdown_distributions")
plt.close(figure)

figure, axis = plt.subplots(figsize=(6.8, 6.0), constrained_layout=True)
axis.scatter(resampled["Maximum drawdown"], resampled["Conditional expected drawdown"], color=PALETTE["green"], s=21, alpha=0.68)
axis.set_xlabel("Maximum drawdown")
axis.set_ylabel("Conditional expected drawdown")
axis.set_title("Two Drawdown Summaries on the Same Resampled Paths")
finish_axis(axis)
save_figure(figure, "resampled_drawdown_scatter")
plt.close(figure)

checks = pd.DataFrame([
    {"Check": "Only design-sample returns were resampled", "Result": len(design_returns) == 5287},
    {"Check": "Requested number of resamples", "Result": len(resampled) == 300},
    {"Check": "All maximum drawdowns lie in [0,1)", "Result": bool(((resampled["Maximum drawdown"] >= 0) & (resampled["Maximum drawdown"] < 1)).all())},
    {"Check": "All conditional expected drawdowns lie in [0,1)", "Result": bool(((resampled["Conditional expected drawdown"] >= 0) & (resampled["Conditional expected drawdown"] < 1)).all())},
    {"Check": "Conditional expected drawdown does not exceed path maximum drawdown", "Result": bool((resampled["Conditional expected drawdown"] <= resampled["Maximum drawdown"] + 1e-12).all())},
])
save_table(checks, "resampling_checks", "Checks for the public-data block resampling experiment.", "tab:chapter22-resampling-checks", fit=True)

assert checks["Result"].all()
assert np.isfinite(resampling_summary.select_dtypes(include=[np.number]).to_numpy()).all()

,Setting,Value
0,Resamples,300
1,Path length,500
2,Block length,20
3,Drawdown horizon,60
4,CED tail level,0.95
5,Random seed,"2,202"


,Measure,Mean,Standard deviation,Coefficient of variation,Minimum,Maximum
0,Maximum drawdown,0.297579,0.111121,0.373417,0.089356,0.707522
1,Conditional expected drawdown,0.238913,0.07226,0.302453,0.089187,0.434212


,Measure,Probability,Quantile
0,Maximum drawdown,0.05,0.141433
1,Maximum drawdown,0.25,0.210862
2,Maximum drawdown,0.5,0.283787
3,Maximum drawdown,0.75,0.373498
4,Maximum drawdown,0.95,0.485313
5,Conditional expected drawdown,0.05,0.132264
6,Conditional expected drawdown,0.25,0.188119
7,Conditional expected drawdown,0.5,0.23103
8,Conditional expected drawdown,0.75,0.284972
9,Conditional expected drawdown,0.95,0.370026


,Check,Result
0,Only design-sample returns were resampled,Pass
1,Requested number of resamples,Pass
2,"All maximum drawdowns lie in [0,1)",Pass
3,"All conditional expected drawdowns lie in [0,1)",Pass
4,Conditional expected drawdown does not exceed ...,Pass
